# 01 · Pipeline walkthrough and agent run

Runs Phase 1 (Bronze → Silver → Gold with quality checks) and Phase 2 (the LangGraph multi-agent system), showing table previews, check results, fault injection and the generated run report — the screenshots required by the project report come from this notebook.

In [ ]:
import os, sys
from pathlib import Path

# Repo root on sys.path (works in a Databricks Git folder and locally)
ROOT = Path(os.getcwd()).resolve().parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd()).resolve()
sys.path.insert(0, str(ROOT))
ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
print("repo root:", ROOT, "| on Databricks:", ON_DATABRICKS)

In [ ]:
if ON_DATABRICKS:
    dbutils.widgets.text("storage_account", "", "ADLS Gen2 account")
    dbutils.widgets.text("secret_scope", "uber-etl", "Secret scope")
    os.environ["STORAGE_MODE"] = "adls"
    os.environ["STORAGE_ACCOUNT_NAME"] = dbutils.widgets.get("storage_account")
    os.environ["DATABRICKS_SECRET_SCOPE"] = dbutils.widgets.get("secret_scope")
    os.environ.setdefault("OPTIMIZE_GOLD", "true")

from src.config import get_config
cfg = get_config()
print("storage mode:", cfg.storage_mode, "| format:", cfg.table_format)
for layer in ("raw", "bronze", "silver", "quarantine", "reports"):
    print(f"{layer:11}", cfg.container_root(layer))

In [ ]:
from src.spark_utils import get_spark, read_table
spark = get_spark(cfg=cfg)

## Phase 1 — Bronze → Silver

In [ ]:
from src import etl_pipeline, gold_analytics, quality_checks as qc
metrics = etl_pipeline.run_bronze_to_silver(spark, cfg)   # pass from_date/to_date for an incremental load
metrics

In [ ]:
display(read_table(spark, cfg.bronze_path, cfg).limit(10)) if ON_DATABRICKS else read_table(spark, cfg.bronze_path, cfg).show(5, truncate=False)

In [ ]:
display(read_table(spark, cfg.silver_path, cfg).limit(10)) if ON_DATABRICKS else read_table(spark, cfg.silver_path, cfg).show(5)

In [ ]:
q = read_table(spark, cfg.quarantine_path, cfg)
q.groupBy("primary_reject_reason").count().orderBy("count", ascending=False).show(truncate=False)

## Data quality checks

In [ ]:
import pandas as pd
checks = qc.run_bronze_checks(spark, cfg) + qc.run_silver_checks(spark, cfg)
pd.DataFrame([{k: c.to_dict()[k] for k in ("layer", "rule", "category", "status", "metric_value", "threshold", "hard")}
              for c in checks])

## Gold analytics

In [ ]:
gold = gold_analytics.build_gold_tables(spark, cfg)
for t in gold["tables"]:
    print(f"--- gold/{t}")
    read_table(spark, cfg.gold_path(t), cfg).show(5)
gold_checks = qc.run_gold_checks(spark, cfg)
pd.DataFrame([{k: c.to_dict()[k] for k in ("rule", "status", "metric_value")} for c in gold_checks])

## Phase 2 — Multi-agent run

The supervisor routes the request; `etl` triggers the Databricks job (or the local runner), `quality` validates Bronze/Silver, `analytics` builds and validates Gold, and the run report is written to `reports/`.

In [ ]:
from src.agents import AgentSystem, export_graph
system = AgentSystem(cfg)
state = system.run("Run the full ETL pipeline and validate every layer")
print(state["status"], state["nodes_run"])
print(state["summary"])

In [ ]:
from IPython.display import Image, Markdown, display as ipy_display
paths = export_graph(system.app, str(ROOT / "docs"))
if "png" in paths: ipy_display(Image(paths["png"]))
ipy_display(Markdown(system.store.read_text(f"runs/{state['run_id']}/report.md")))

## Ask the explain agent

In [ ]:
answer = system.run("Why did the last run end with warnings, and what should I do?")
print(answer["answer"])

## Fault injection (local mode)

Corrupt the landing zone and watch the agents stop at the right node. `scripts/evaluate_agents.py` runs every scenario and computes detection / false-alarm / recovery rates.

In [ ]:
from src.fault_injection import FAULTS
pd.DataFrame([{"fault": f.name, "description": f.description, "expected": f.expected_status,
               "stops_at": f.expected_stop_node, "rules": ", ".join(f.expected_rules)} for f in FAULTS.values()])